### Prepare config file

Approved Drugs Data has been downloaded from Drugbank (https://go.drugbank.com/releases/latest#open-data). Now we are going to convert it to csv file to run `DORAnet` for all the possible substructures. 

In [27]:
import csv
import pandas as pd
from pathlib import Path
import re
from rdkit import Chem
from rdkit.Chem import PandasTools
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

### Load the approved Drug molecules

In [28]:
drugBankDrugDataDF = pd.read_csv("drugbank_vocabulary.csv", dtype=str)
print(f"Loaded {len(drugBankDrugDataDF)} molecules")
drugBankDrugDataDF

Loaded 19830 molecules


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key
0,DB00001,BTD00024 | BIOD00024,Lepirudin,138068-37-8,Y43GF64R34,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru...",NaN
1,DB00002,BTD00071 | BIOD00071,Cetuximab,205923-56-4,PQX0D8J21J,Cetuximab | Cétuximab | Cetuximabum | Chimeric...,NaN
2,DB00003,BTD00001 | BIOD00001,Dornase alfa,143831-71-4,953A26OA1Y,Deoxyribonuclease (human clone 18-1 protein mo...,NaN
3,DB00004,BTD00084 | BIOD00084,Denileukin diftitox,173146-27-5,25E79B5CTM,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...,NaN
4,DB00005,BTD00052 | BIOD00052,Etanercept,185243-69-0,OP401G7OJC,Etanercept | etanercept-szzs | etanercept-ykro...,NaN
...,...,...,...,...,...,...,...
19825,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N
19826,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N
19827,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N
19828,DB31795,NaN,FRAX716,1432908-40-1,NaN,NaN,NaN


### Read Drugbank structure file data (.sdf) format

In [29]:
structureFilePath = "open_structures.sdf"

# RDKit helper that loads SDF into a DataFrame (keeps an RDKit Mol column)
drugBankDrugStructureDataDF = PandasTools.LoadSDF(
    structureFilePath,
    smilesName="smiles",      # adds SMILES column
    molColName="mol",         # RDKit Mol objects
    includeFingerprints=False,
    embedProps=True           # include SDF properties as columns
)

print(f"Loaded {len(drugBankDrugStructureDataDF)} molecules")
drugBankDrugStructureDataDF

[15:27:46] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 36 ignored
[15:27:46] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[15:27:46] ERROR: Could not sanitize molecule ending on line 127274
[15:27:46] ERROR: Explicit valence for atom # 13 Cl, 5, is greater than permitted
[15:27:47] Explicit valence for atom # 19 O, 3, is greater than permitted
[15:27:47] ERROR: Could not sanitize molecule ending on line 173531
[15:27:47] ERROR: Explicit valence for atom # 19 O, 3, is greater than permitted
[15:27:47] Explicit valence for atom # 15 O, 3, is greater than permitted
[15:27:47] ERROR: Could not sanitize molecule ending on line 264332
[15:27:47] ERROR: Explicit valence for atom # 15 O, 3, is greater than permitted
[15:27:48] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 16 ignored
[15:27:48] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 20 ignored.
[15:27:48] Warning: ambiguous stereo

Loaded 14612 molecules


,DRUGBANK_ID,SECONDARY_ACCESSION_NUMBERS,COMMON_NAME,CAS_NUMBER,UNII,SYNONYMS,ID,smiles,mol
0,DB00006,BTD00076; EXPT03302; BIOD00076; DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin; Bivalirudina; Bivalirudinum,,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,<rdkit.Chem.rdchem.Mol object at 0x7fe80784b300>
1,DB00014,BTD00113; BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin; Goserelina,[NO NAME],CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,<rdkit.Chem.rdchem.Mol object at 0x7fe792f73760>
2,DB00027,BTD00036; BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D; Gramicidin; Gram...,,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,<rdkit.Chem.rdchem.Mol object at 0x7fe792f73a00>
3,DB00035,BTD00112; BTD00061; BIOD00112; BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin; 1-(3-mercap...,,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,<rdkit.Chem.rdchem.Mol object at 0x7fe792f73990>
4,DB00050,BTD00115; APRD00686; BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix; Cetrorelixum,,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,<rdkit.Chem.rdchem.Mol object at 0x7fe792f73a70>
...,...,...,...,...,...,...,...,...,...
14614,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline; O,n-dipalmitoylhyd...",,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324eea0>
14615,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid; (r)-gossypol acetic ...,,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324ef10>
14616,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid; (±)-gossypol aceti...,,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324eff0>
14617,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324f060>


In [30]:
# Merge on DrugBank ID first
mergedOnDrugBankID = drugBankDrugDataDF.merge(
    drugBankDrugStructureDataDF[['DRUGBANK_ID', 'smiles', 'mol']],
    left_on='DrugBank ID',
    right_on='DRUGBANK_ID',
    how='left',
    suffixes=('', '_fromID')
)
mergedOnDrugBankID = mergedOnDrugBankID.drop(columns=['DRUGBANK_ID'])

# For rows still missing SMILES, try merging on CAS number
missingMask = mergedOnDrugBankID['smiles'].isna()
missingCount = missingMask.sum()
print(f"Matched by DrugBank ID: {len(mergedOnDrugBankID) - missingCount}")
print(f"Still missing after DrugBank ID match: {missingCount}")

if missingCount > 0:
    # Clean CAS columns for matching
    drugBankDrugDataDF_clean = mergedOnDrugBankID[missingMask].copy()
    drugBankDrugDataDF_clean['CAS_clean'] = drugBankDrugDataDF_clean['CAS'].astype(str).str.strip()

    structureDFClean = drugBankDrugStructureDataDF[['CAS_NUMBER', 'smiles', 'mol']].copy()
    structureDFClean['CAS_clean'] = structureDFClean['CAS_NUMBER'].astype(str).str.strip()
    structureDFClean = structureDFClean[
        (structureDFClean['CAS_clean'] != '') &
        (structureDFClean['CAS_clean'] != 'nan')
    ].drop_duplicates(subset='CAS_clean')

    casMerged = drugBankDrugDataDF_clean.merge(
        structureDFClean[['CAS_clean', 'smiles', 'mol']],
        on='CAS_clean',
        how='left',
        suffixes=('', '_fromCAS')
    )

    # Fill missing smiles and mol from CAS match
    mergedOnDrugBankID.loc[missingMask, 'smiles'] = casMerged['smiles_fromCAS'].values
    mergedOnDrugBankID.loc[missingMask, 'mol'] = casMerged['mol_fromCAS'].values

    casMatchCount = casMerged['smiles_fromCAS'].notna().sum()
    print(f"Matched by CAS number: {casMatchCount}")

# Drop helper columns if present
dropCols = [c for c in mergedOnDrugBankID.columns if c.endswith('_clean')]
mergedOnDrugBankID = mergedOnDrugBankID.drop(columns=dropCols, errors='ignore')

drugBankDrugDataDF = mergedOnDrugBankID.copy()

# Summary
totalDrugs = len(drugBankDrugDataDF)
foundCount = drugBankDrugDataDF['smiles'].notna().sum()
missingCount = drugBankDrugDataDF['smiles'].isna().sum()

print(f"\nFinal Summary:")
print(f"  Total drugs: {totalDrugs}")
print(f"  With SMILES: {foundCount} ({foundCount/totalDrugs*100:.2f}%)")
print(f"  Missing SMILES: {missingCount} ({missingCount/totalDrugs*100:.2f}%)")

drugBankDrugDataDF

Matched by DrugBank ID: 14612
Still missing after DrugBank ID match: 5218
Matched by CAS number: 6

Final Summary:
  Total drugs: 19830
  With SMILES: 14618 (73.72%)
  Missing SMILES: 5212 (26.28%)


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,smiles,mol
0,DB00001,BTD00024 | BIOD00024,Lepirudin,138068-37-8,Y43GF64R34,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru...",NaN,NaN,NaN
1,DB00002,BTD00071 | BIOD00071,Cetuximab,205923-56-4,PQX0D8J21J,Cetuximab | Cétuximab | Cetuximabum | Chimeric...,NaN,NaN,NaN
2,DB00003,BTD00001 | BIOD00001,Dornase alfa,143831-71-4,953A26OA1Y,Deoxyribonuclease (human clone 18-1 protein mo...,NaN,NaN,NaN
3,DB00004,BTD00084 | BIOD00084,Denileukin diftitox,173146-27-5,25E79B5CTM,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...,NaN,NaN,NaN
4,DB00005,BTD00052 | BIOD00052,Etanercept,185243-69-0,OP401G7OJC,Etanercept | etanercept-szzs | etanercept-ykro...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
19825,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324ef10>
19826,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324eff0>
19827,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,<rdkit.Chem.rdchem.Mol object at 0x7fe79324f060>
19828,DB31795,NaN,FRAX716,1432908-40-1,NaN,NaN,NaN,NaN,NaN


In [31]:
# Split into with and without SMILES
drugBankDrugDataDF_wSMILES = drugBankDrugDataDF[drugBankDrugDataDF['smiles'].notna()].copy()
drugBankDrugDataDF_woSMILES = drugBankDrugDataDF[drugBankDrugDataDF['smiles'].isna()].copy()

# Validate SMILES with RDKit and keep only valid ones
drugBankDrugDataDF_wSMILES['Canonical_SMILES'] = drugBankDrugDataDF_wSMILES['smiles'].apply(
    lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(str(smi).strip()), canonical=True, isomericSmiles=True)
    if Chem.MolFromSmiles(str(smi).strip()) is not None else None
)

# Remove rows where RDKit could not parse the SMILES
invalidMask = drugBankDrugDataDF_wSMILES['Canonical_SMILES'].isna()
invalidCount = invalidMask.sum()

if invalidCount > 0:
    # Move invalid SMILES rows to woSMILES
    drugBankDrugDataDF_woSMILES = pd.concat([
        drugBankDrugDataDF_woSMILES,
        drugBankDrugDataDF_wSMILES[invalidMask]
    ], ignore_index=True)

    drugBankDrugDataDF_wSMILES = drugBankDrugDataDF_wSMILES[~invalidMask].copy()

# Drop old smiles and mol columns from wSMILES
drugBankDrugDataDF_wSMILES = drugBankDrugDataDF_wSMILES.drop(columns=['smiles', 'mol'], errors='ignore')

# Drop smiles, mol, and Canonical_SMILES columns from woSMILES (keep all other original columns)
drugBankDrugDataDF_woSMILES = drugBankDrugDataDF_woSMILES.drop(
    columns=['smiles', 'mol', 'Canonical_SMILES'], errors='ignore'
)

# Reset indices
drugBankDrugDataDF_wSMILES = drugBankDrugDataDF_wSMILES.reset_index(drop=True)
drugBankDrugDataDF_woSMILES = drugBankDrugDataDF_woSMILES.reset_index(drop=True)

# Summary
totalDrugs = len(drugBankDrugDataDF)
validCount = len(drugBankDrugDataDF_wSMILES)
missingCount = len(drugBankDrugDataDF_woSMILES)

print(f"Total drugs: {totalDrugs}")
print(f"With valid SMILES (drugBankDrugDataDF_wSMILES): {validCount} ({validCount/totalDrugs*100:.2f}%)")
print(f"Without SMILES (drugBankDrugDataDF_woSMILES): {missingCount} ({missingCount/totalDrugs*100:.2f}%)")
if invalidCount > 0:
    print(f"  (includes {invalidCount} drugs with unparseable SMILES)")

[15:28:01] Unusual charge on atom 42 number of radical electrons set to zero
[15:28:01] Unusual charge on atom 42 number of radical electrons set to zero
[15:28:03] WARNING: not removing hydrogen atom without neighbors
[15:28:03] WARNING: not removing hydrogen atom without neighbors
[15:28:03] WARNING: not removing hydrogen atom without neighbors
[15:28:03] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[15:28:04] WARNING: not removing hydrogen atom without neighbors
[

Total drugs: 19830
With valid SMILES (drugBankDrugDataDF_wSMILES): 14618 (73.72%)
Without SMILES (drugBankDrugDataDF_woSMILES): 5212 (26.28%)


In [20]:
drugBankDrugDataDF_wSMILES.to_csv('drugBankDrugDataDF_wSMILES.csv', index=False, encoding="utf-8")

In [32]:
drugBankDrugDataDF_wSMILES

,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES
0,DB00006,BTD00076 | EXPT03302 | BIOD00076 | DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin | Bivalirudina | Bivalirudinum,OIRCOABEOLEUMC-GEJPAHFPSA-N,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...
1,DB00014,BTD00113 | BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin | Goserelina,BLCLNMBMMGCOAS-URPVMXJPSA-N,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,DB00027,BTD00036 | BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D | Gramicidin | Gr...,NDAYQJDHGXTBJL-MWWSRJDJSA-N,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...
4,DB00050,BTD00115 | APRD00686 | BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix | Cetrorelixum,SBNPWPIBESPSIF-MHWMIDJBSA-N,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...
...,...,...,...,...,...,...,...,...
14613,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...
14614,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14615,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14616,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...


In [35]:
drugBankDrugDataDF_woSMILES.columns

Index(['DrugBank ID', 'Accession Numbers', 'Common name', 'CAS', 'UNII',
       'Synonyms', 'Standard InChI Key'],
      dtype='object')

### This code searches for SMILES using 8 different strategies across 4 public databases in a cascading order:

- Databases used: PubChem, ChEMBL, NCI Chemical Identifier Resolver (CIR), UniChem (EBI).
- Identifiers tried: InChI Key, drug name, CAS number, UNII code

In [ ]:
print(f"Loaded {len(drugBankDrugDataDF_woSMILES)} drugs from DrugBank (without SMILES)")

# =====================================================================
# Session with connection pooling for faster requests
# =====================================================================
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(
    pool_connections=20,
    pool_maxsize=20,
    max_retries=2
)
session.mount('https://', adapter)
session.mount('http://', adapter)


# =====================================================================
# API Functions using session for connection reuse
# =====================================================================

def getSmilesFromPubChemByName(drugName: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(drugName)}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByInChIKey(inchiKey: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchiKey}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByCAS(cas: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromPubChemByUNII(unii: str) -> str | None:
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{unii}/property/CanonicalSMILES/JSON"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None


def getSmilesFromCIRByName(drugName: str) -> str | None:
    try:
        url = f"https://cactus.nci.nih.gov/chemical/structure/{requests.utils.quote(drugName)}/smiles"
        response = session.get(url, timeout=10)
        if response.status_code == 200 and len(response.text) < 1000 and '\n' not in response.text.strip():
            return response.text.strip()
    except Exception:
        pass
    return None


def getSmilesFromCIRByCAS(cas: str) -> str | None:
    try:
        url = f"https://cactus.nci.nih.gov/chemical/structure/{cas}/smiles"
        response = session.get(url, timeout=10)
        if response.status_code == 200 and len(response.text) < 1000 and '\n' not in response.text.strip():
            return response.text.strip()
    except Exception:
        pass
    return None


def getSmilesFromUniChemByInChIKey(inchiKey: str) -> str | None:
    try:
        url = f"https://www.ebi.ac.uk/unichem/rest/inchikey/{inchiKey}"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                srcCompoundId = data[0].get('src_compound_id')
                if srcCompoundId:
                    return getSmilesFromPubChemByName(srcCompoundId)
    except Exception:
        pass
    return None


def getSmilesFromChEMBLByName(drugName: str) -> str | None:
    try:
        url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search?q={requests.utils.quote(drugName)}&format=json"
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            molecules = data.get('molecules', [])
            if molecules and molecules[0].get('molecule_structures'):
                return molecules[0]['molecule_structures'].get('canonical_smiles')
    except Exception:
        pass
    return None


def canonicalize(smiles: str) -> str | None:
    if smiles is None:
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def fetchSmiles(row) -> tuple[str | None, str]:
    drugName = str(row.get('Common name', '')).strip()
    inchiKey = str(row.get('Standard InChI Key', '')).strip()
    cas = str(row.get('CAS', '')).strip()
    unii = str(row.get('UNII', '')).strip()

    if drugName.lower() == 'nan' or drugName == '':
        drugName = None
    if inchiKey.lower() == 'nan' or inchiKey == '':
        inchiKey = None
    if cas.lower() == 'nan' or cas == '':
        cas = None
    if unii.lower() == 'nan' or unii == '':
        unii = None

    if inchiKey:
        smiles = getSmilesFromPubChemByInChIKey(inchiKey)
        if smiles:
            return canonicalize(smiles), "PubChem_InChIKey"

    if drugName:
        smiles = getSmilesFromPubChemByName(drugName)
        if smiles:
            return canonicalize(smiles), "PubChem_Name"

    if cas:
        smiles = getSmilesFromPubChemByCAS(cas)
        if smiles:
            return canonicalize(smiles), "PubChem_CAS"

    if unii:
        smiles = getSmilesFromPubChemByUNII(unii)
        if smiles:
            return canonicalize(smiles), "PubChem_UNII"

    if drugName:
        smiles = getSmilesFromChEMBLByName(drugName)
        if smiles:
            return canonicalize(smiles), "ChEMBL_Name"

    if drugName:
        smiles = getSmilesFromCIRByName(drugName)
        if smiles:
            return canonicalize(smiles), "CIR_Name"

    if cas:
        smiles = getSmilesFromCIRByCAS(cas)
        if smiles:
            return canonicalize(smiles), "CIR_CAS"

    if inchiKey:
        smiles = getSmilesFromUniChemByInChIKey(inchiKey)
        if smiles:
            return canonicalize(smiles), "UniChem_InChIKey"

    return None, "NotFound"


def fetchSmilesWorker(args):
    """Wrapper for thread pool - uses positional index instead of df index."""
    posIdx, row = args
    smiles, source = fetchSmiles(row)
    drugName = str(row.get('Common name', 'Unknown')).strip()
    return posIdx, smiles, source, drugName


# =====================================================================
# Fetch SMILES using thread pool for parallel API calls
# =====================================================================

# Reset index to ensure sequential indexing
drugBankDrugDataDF_woSMILES = drugBankDrugDataDF_woSMILES.reset_index(drop=True)

totalDrugs = len(drugBankDrugDataDF_woSMILES)
numWorkers = 8

print(f"\nFetching SMILES for {totalDrugs} drugs using {numWorkers} parallel threads...")
print(f"APIs: PubChem, ChEMBL, CIR, UniChem")
print(f"Identifiers: InChI Key, Drug Name, CAS, UNII")
print("-" * 70)

# Prepare tasks using positional index
tasks = [(idx, row) for idx, row in drugBankDrugDataDF_woSMILES.iterrows()]

# Initialize result storage
smilesResults = [None] * totalDrugs
sourceResults = ['NotFound'] * totalDrugs
failedDrugs = []
completedCount = 0

# Run with thread pool
with ThreadPoolExecutor(max_workers=numWorkers) as executor:
    futures = {executor.submit(fetchSmilesWorker, task): task[0] for task in tasks}

    for future in as_completed(futures):
        posIdx, smiles, source, drugName = future.result()
        smilesResults[posIdx] = smiles
        sourceResults[posIdx] = source

        if smiles is None:
            failedDrugs.append(drugName)

        completedCount += 1
        if completedCount % 100 == 0:
            successCount = sum(1 for s in smilesResults if s is not None)
            print(f"  Processed {completedCount}/{totalDrugs} | "
                  f"Found: {successCount} | "
                  f"Missing: {len(failedDrugs)}")

# Add columns
drugBankDrugDataDF_woSMILES['Canonical_SMILES'] = smilesResults
drugBankDrugDataDF_woSMILES['SMILES_Source'] = sourceResults

# =====================================================================
# Summary
# =====================================================================

foundCount = drugBankDrugDataDF_woSMILES['Canonical_SMILES'].notna().sum()
missingCount = drugBankDrugDataDF_woSMILES['Canonical_SMILES'].isna().sum()
foundPercent = (foundCount / totalDrugs) * 100
missingPercent = (missingCount / totalDrugs) * 100

print(f"\n{'=' * 70}")
print(f"SMILES Retrieval Summary:")
print(f"{'=' * 70}")
print(f"  Found:   {foundCount} ({foundPercent:.2f}%)")
print(f"  Missing: {missingCount} ({missingPercent:.2f}%)")

print(f"\nSource breakdown:")
sourceCounts = drugBankDrugDataDF_woSMILES['SMILES_Source'].value_counts()
for source, count in sourceCounts.items():
    print(f"  {source}: {count} ({count/totalDrugs*100:.2f}%)")

if failedDrugs:
    print(f"\nFailed drugs (first 20):")
    for drug in failedDrugs[:20]:
        print(f"  - {drug}")
    if len(failedDrugs) > 20:
        print(f"  ... and {len(failedDrugs) - 20} more")

drugBankDrugDataDF_woSMILES

Loaded 5212 drugs from DrugBank (without SMILES)

Fetching SMILES for 5212 drugs using 8 parallel threads...
APIs: PubChem, ChEMBL, CIR, UniChem
Identifiers: InChI Key, Drug Name, CAS, UNII
----------------------------------------------------------------------


[15:56:10] SMILES Parse Error: syntax error while parsing: N|[Co+3](|N)(|N)(|N)(|N)|N
[15:56:10] SMILES Parse Error: Failed parsing SMILES 'N|[Co+3](|N)(|N)(|N)(|N)|N' for input: 'N|[Co+3](|N)(|N)(|N)(|N)|N'
[16:08:22] SMILES Parse Error: syntax error while parsing: [Os++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27
[16:08:22] SMILES Parse Error: Failed parsing SMILES '[Os++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27' for input: '[Os++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27'


  Processed 100/5212 | Found: 16 | Missing: 84


[16:40:06] SMILES Parse Error: syntax error while parsing: [Ru++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27
[16:40:06] SMILES Parse Error: Failed parsing SMILES '[Ru++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27' for input: '[Ru++]|1|2(|N3=CC=NC3)(|n4ccccc4c5ccccn|15)|n6ccccc6c7ccccn|27'


  Processed 200/5212 | Found: 45 | Missing: 155


[16:51:26] SMILES Parse Error: syntax error while parsing: [Fe+4]|1|2|3|[N-]4C5=Cc6[n-]|1c(C=C7[N-]|2C(=Cc8[n-]|3c(C=C4C(=C5C)CCC(=O)OC)c(CCC(=O)OC)c8C)C(=C7C)C=C)c(C=C)c6C
[16:51:26] SMILES Parse Error: Failed parsing SMILES '[Fe+4]|1|2|3|[N-]4C5=Cc6[n-]|1c(C=C7[N-]|2C(=Cc8[n-]|3c(C=C4C(=C5C)CCC(=O)OC)c(CCC(=O)OC)c8C)C(=C7C)C=C)c(C=C)c6C' for input: '[Fe+4]|1|2|3|[N-]4C5=Cc6[n-]|1c(C=C7[N-]|2C(=Cc8[n-]|3c(C=C4C(=C5C)CCC(=O)OC)c(CCC(=O)OC)c8C)C(=C7C)C=C)c(C=C)c6C'


  Processed 300/5212 | Found: 68 | Missing: 232
  Processed 400/5212 | Found: 126 | Missing: 274


[17:22:42] Explicit valence for atom # 0 O, 3, is greater than permitted
[17:25:47] SMILES Parse Error: syntax error while parsing: [F]|[Al](|[F])(|[F])|[F-]
[17:25:47] SMILES Parse Error: Failed parsing SMILES '[F]|[Al](|[F])(|[F])|[F-]' for input: '[F]|[Al](|[F])(|[F])|[F-]'


  Processed 500/5212 | Found: 184 | Missing: 316
  Processed 600/5212 | Found: 245 | Missing: 355


[17:56:06] SMILES Parse Error: syntax error while parsing: [Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(CCC(O)=O)cc5N|2C=C6C=CC=CC6=O|3
[17:56:06] SMILES Parse Error: Failed parsing SMILES '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(CCC(O)=O)cc5N|2C=C6C=CC=CC6=O|3' for input: '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(CCC(O)=O)cc5N|2C=C6C=CC=CC6=O|3'
[17:56:08] SMILES Parse Error: syntax error while parsing: [Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(cc5N|2C=C6C=CC=CC6=O|3)C(O)=O
[17:56:08] SMILES Parse Error: Failed parsing SMILES '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(cc5N|2C=C6C=CC=CC6=O|3)C(O)=O' for input: '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccc(cc5N|2C=C6C=CC=CC6=O|3)C(O)=O'
[17:56:08] SMILES Parse Error: syntax error while parsing: [Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccccc5N|2C=C6C=CC=CC6=O|3
[17:56:08] SMILES Parse Error: Failed parsing SMILES '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccccc5N|2C=C6C=CC=CC6=O|3' for input: '[Fe]|1|2|3|O=C4C=CC=CC4=CN|1c5ccccc5N|2C=C6C=CC=CC6=O|3'


### Extract drug data from full `DrugBank` data set

In [ ]:
import xml.etree.ElementTree as ET

xmlPath = "full_DrugBank_database.xml" 

def stripNamespace(tag: str) -> str:
    # '{namespace}tag' -> 'tag'
    return tag.split("}", 1)[-1] if "}" in tag else tag

rows = []

# iterparse streams the file (good for very large DrugBank XML)
context = ET.iterparse(xmlPath, events=("end",))

for event, elem in context:
    if stripNamespace(elem.tag) != "drug":
        continue

    # --- Extract fields from one <drug> element ---
    drugType = elem.attrib.get("type", None)

    # DrugBank primary ID (first <drugbank-id primary="true">)
    primaryId = None
    for dbid in elem.findall("./{*}drugbank-id"):
        if dbid.attrib.get("primary") == "true":
            primaryId = dbid.text
            break
    if primaryId is None:
        # fallback: first id
        firstId = elem.find("./{*}drugbank-id")
        primaryId = firstId.text if firstId is not None else None

    nameElem = elem.find("./{*}name")
    drugName = nameElem.text if nameElem is not None else None

    casElem = elem.find("./{*}cas-number")
    casNumber = casElem.text if casElem is not None else None

    # groups: <groups><group>approved</group>...
    groups = [g.text for g in elem.findall("./{*}groups/{*}group") if g.text]
    groupsStr = "|".join(groups) if groups else None

    # ATC codes: <atc-codes><atc-code code="A01AB"/></atc-codes>
    atcCodes = [a.attrib.get("code") for a in elem.findall("./{*}atc-codes/{*}atc-code") if a.attrib.get("code")]
    atcCodesStr = "|".join(atcCodes) if atcCodes else None

    # synonyms: <synonyms><synonym ...>...</synonym></synonyms>
    synonyms = [s.text for s in elem.findall("./{*}synonyms/{*}synonym") if s.text]
    synonymsStr = "|".join(synonyms) if synonyms else None

    rows.append({
        "drugbankId": primaryId,
        "name": drugName,
        "type": drugType,
        "groups": groupsStr,
        "casNumber": casNumber,
        "atcCodes": atcCodesStr,
        "synonyms": synonymsStr,
    })

    # free memory for processed element
    elem.clear()

fullDrugBankDrugDataDF = pd.DataFrame(rows)

print(f"Loaded {len(fullDrugBankDrugDataDF)} drugs")
fullDrugBankDrugDataDF

### For a broader search space take `targetSmiles` from a data frame

In [23]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = pd.read_csv("drugBankDrugDataDF_wSMILES.csv")
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue
    
    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Print suggested max_atoms config (max values + 50% increase)
maxC = startersDF['C'].max()
maxN = startersDF['N'].max()
maxO = startersDF['O'].max()
maxS = startersDF['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))} # Carbon")
print(f"  N: {int(np.ceil(maxN * 1.5))} # Nitrogen")
print(f"  O: {int(np.ceil(maxO * 1.5))} # Oxygen")
print(f"  S: {int(np.ceil(maxS * 1.5))} # Sulfur")

startersDF

Loaded 14618 molecules


[14:43:50] Unusual charge on atom 42 number of radical electrons set to zero
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors
[14:43:51] WARNING: not removing hydrogen atom without neighbors



Atom count ranges across 14618 molecules:
  C (Carbon):   min = 0, max = 208
  N (Nitrogen): min = 0, max = 68
  O (Oxygen):   min = 0, max = 110
  S (Sulfur):   min = 0, max = 18

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 312 # Carbon
  N: 102 # Nitrogen
  O: 165 # Oxygen
  S: 27 # Sulfur


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES,C,N,O,S
0,DB00006,BTD00076 | EXPT03302 | BIOD00076 | DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin | Bivalirudina | Bivalirudinum,OIRCOABEOLEUMC-GEJPAHFPSA-N,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,98,24,33,0
1,DB00014,BTD00113 | BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin | Goserelina,BLCLNMBMMGCOAS-URPVMXJPSA-N,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,59,18,14,0
2,DB00027,BTD00036 | BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D | Gramicidin | Gr...,NDAYQJDHGXTBJL-MWWSRJDJSA-N,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,96,19,16,0
3,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,46,14,12,2
4,DB00050,BTD00115 | APRD00686 | BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix | Cetrorelixum,SBNPWPIBESPSIF-MHWMIDJBSA-N,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,70,17,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14613,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,37,1,5,0
14614,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0
14615,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0
14616,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,29,7,1,1


### Create a data frame with small number of compounds

In [44]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = pd.read_csv("drugBankDrugDataDF_wSMILES.csv")
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges for all molecules
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Filter: all atom counts must be less than 50
atomLimit = 25
startersDF_short = startersDF[
    (startersDF['C'] < atomLimit) &
    (startersDF['N'] < atomLimit) &
    (startersDF['O'] < atomLimit) &
    (startersDF['S'] < atomLimit)
].copy().reset_index(drop=True)

removedCount = len(startersDF) - len(startersDF_short)
print(f"\nFiltering to molecules with C, N, O, S each < {atomLimit}:")
print(f"  Before: {len(startersDF)}")
print(f"  After:  {len(startersDF_short)}")
print(f"  Removed: {removedCount} ({removedCount/len(startersDF)*100:.2f}%)")

# Print atom count ranges for filtered molecules
print(f"\nAtom count ranges after filtering ({len(startersDF_short)} molecules):")
print(f"  C (Carbon):   min = {startersDF_short['C'].min()}, max = {startersDF_short['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF_short['N'].min()}, max = {startersDF_short['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF_short['O'].min()}, max = {startersDF_short['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF_short['S'].min()}, max = {startersDF_short['S'].max()}")

# Print suggested max_atoms config based on filtered molecules
maxC = startersDF_short['C'].max()
maxN = startersDF_short['N'].max()
maxO = startersDF_short['O'].max()
maxS = startersDF_short['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))}   # Carbon (max in data: {maxC})")
print(f"  N: {int(np.ceil(maxN * 1.5))}   # Nitrogen (max in data: {maxN})")
print(f"  O: {int(np.ceil(maxO * 1.5))}   # Oxygen (max in data: {maxO})")
print(f"  S: {int(np.ceil(maxS * 1.5))}   # Sulfur (max in data: {maxS})")

# Drop atom count columns before saving
startersDF_short = startersDF_short.drop(columns=['C', 'N', 'O', 'S'])

# Save filtered CSV
startersDF_short.to_csv("drugBankDrugDataDF_wSMILES_short.csv", index=False)
print(f"\nSaved {len(startersDF_short)} molecules to drugBankDrugDataDF_wSMILES_short.csv")

startersDF_short

Loaded 14618 molecules


[11:55:30] Unusual charge on atom 42 number of radical electrons set to zero
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors
[11:55:31] WARNING: not removing hydrogen atom without neighbors



Atom count ranges across 14618 molecules:
  C (Carbon):   min = 0, max = 208
  N (Nitrogen): min = 0, max = 68
  O (Oxygen):   min = 0, max = 110
  S (Sulfur):   min = 0, max = 18

Filtering to molecules with C, N, O, S each < 25:
  Before: 14618
  After:  11798
  Removed: 2820 (19.29%)

Atom count ranges after filtering (11798 molecules):
  C (Carbon):   min = 0, max = 24
  N (Nitrogen): min = 0, max = 18
  O (Oxygen):   min = 0, max = 24
  S (Sulfur):   min = 0, max = 8

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 36   # Carbon (max in data: 24)
  N: 27   # Nitrogen (max in data: 18)
  O: 36   # Oxygen (max in data: 24)
  S: 12   # Sulfur (max in data: 8)

Saved 11798 molecules to drugBankDrugDataDF_wSMILES_short.csv


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES
0,DB00114,NUTR00045,Pyridoxal phosphate,54-47-7,0ABW0CG2BM,3-hydroxy-2-methyl-5-((phosphonooxy)methyl)-4-...,NGVDGCNFYWLIFO-UHFFFAOYSA-N,Cc1ncc(COP(=O)(O)O)c(C=O)c1O
1,DB00116,NUTR00056,Tetrahydrofolic acid,135-16-0,43ZWB253H4,"5,6,7,8-tetrahydrofolate | 5,6,7,8-tetrahydrof...",MSTNYGQPCMXVAQ-KIYNQFGBSA-N,Nc1nc(=O)c2c([nH]1)NCC(CNc1ccc(C(=O)N[C@@H](CC...
2,DB00117,NUTR00030,Histidine,71-00-1,4QD397987E,(S)-4-(2-Amino-2-carboxyethyl)imidazole | (S)-...,HNDVDQJCIGZPNO-YFKPBYRVSA-N,N[C@@H](Cc1c[nH]cn1)C(=O)O
3,DB00118,NUTR00052,Ademetionine,29908-03-0,7LP2MPO46S,Ademetionine | AdoMet | L-S-Adenosylmethionine...,MEFKEPWMEQBLKI-AIRLBKTGSA-N,C[S+](CC[C@H](N)C(=O)[O-])C[C@H]1O[C@@H](n2cnc...
4,DB00119,NUTR00050 | DB11194,Pyruvic acid,127-17-3,8558G7RUTR,2-ketopropionic acid | 2-oxopropanoic acid | 2...,LCTONWCANYUPML-UHFFFAOYSA-N,CC(=O)C(=O)O
...,...,...,...,...,...,...,...,...
11793,DB21749,NaN,WX-554,945529-91-9,36TRH66AA6,NaN,DJIBEZXMCYNIQJ-UHFFFAOYSA-N,NCC1CN(C(=O)c2c(Nc3ccc(I)cc3F)sc3ncccc23)C1
11794,DB21752,DB24221,Tersolisib,2883540-92-7,AGX9NKC8M9,Tersolisib,LGPNQALKGDDVBD-CYBMUJFWSA-N,Cc1c([C@@H](NC(=O)Nc2cnc(N)nc2)C(F)(F)F)oc2c(F...
11795,DB21759,NaN,Afloqualone,56287-74-2,CO4U2C8ORZ,Aflocualona | Afloqualon | Afloqualone,VDOSWXIDETXFET-UHFFFAOYSA-N,Cc1ccccc1-n1c(CF)nc2ccc(N)cc2c1=O
11796,DB21769,NaN,Trineumin,NaN,X5ME2L52ZV,4-(4-(2-(Diethylamino)ethoxy)phenyl)-1-(4-meth...,CGZIGMPINMFIFS-UHFFFAOYSA-N,CCN(CC)CCOc1ccc(-c2nnn(Cc3ccc(OC)cc3)c2N)cc1


### Print `Canonical_SMILES`

In [ ]:
from rdkit import Chem
from collections import Counter
import numpy as np

startersDF_short_SMILESonly = pd.read_csv("drugBankDrugDataDF_wSMILES_short.csv")
print(f"Total molecules: {len(startersDF_short_SMILESonly)}\n")

print(f"{'Idx':<5} {'C':>4} {'N':>4} {'O':>4} {'S':>4}   {'Canonical_SMILES'}")
print("-" * 100)

for idx, smi in enumerate(startersDF_short_SMILESonly['Canonical_SMILES']):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{idx:<5} {'?':>4} {'?':>4} {'?':>4} {'?':>4}   {smi}")
        continue

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    c = atomCounter.get('C', 0)
    n = atomCounter.get('N', 0)
    o = atomCounter.get('O', 0)
    s = atomCounter.get('S', 0)

    sugC = int(np.ceil(c * 1.5))
    sugN = int(np.ceil(n * 1.5))
    sugO = int(np.ceil(o * 1.5))
    sugS = int(np.ceil(s * 1.5))

    sugC = max(sugC, 1)
    sugN = max(sugN, 1)
    sugO = max(sugO, 1)
    sugS = max(sugS, 1)

    print(f"{idx:<5} {c:>4} {n:>4} {o:>4} {s:>4}   {smi}")
    print(f"{'':>5} max_atoms: C:{sugC}, N:{sugN}, O:{sugO}, S:{sugS}")

In [41]:
from rdkit import Chem
from collections import Counter
import numpy as np

startersDF_short_SMILESonly = pd.read_csv("drugBankDrugDataDF_wSMILES_short.csv")
print(f"Total molecules: {len(startersDF_short_SMILESonly)}\n")

atomData = []

for smi in startersDF_short_SMILESonly['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomData.append({
            'C': 0, 'N': 0, 'O': 0, 'S': 0,
            'Suggested_C': 1, 'Suggested_N': 1, 'Suggested_O': 1, 'Suggested_S': 1
        })
        continue

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    c = atomCounter.get('C', 0)
    n = atomCounter.get('N', 0)
    o = atomCounter.get('O', 0)
    s = atomCounter.get('S', 0)

    atomData.append({
        'C': c,
        'N': n,
        'O': o,
        'S': s,
        'Suggested_C': max(int(np.ceil(c * 1.5)), 1),
        'Suggested_N': max(int(np.ceil(n * 1.5)), 1),
        'Suggested_O': max(int(np.ceil(o * 1.5)), 1),
        'Suggested_S': max(int(np.ceil(s * 1.5)), 1)
    })

atomDataDF = pd.DataFrame(atomData)
startersDF_short_SMILESonly = pd.concat(
    [startersDF_short_SMILESonly, atomDataDF], axis=1
)

startersDF_short_SMILESonly

Total molecules: 14332



[11:52:18] Unusual charge on atom 42 number of radical electrons set to zero
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors
[11:52:19] WARNING: not removing hydrogen atom without neighbors


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES,C,N,O,S,Suggested_C,Suggested_N,Suggested_O,Suggested_S
0,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,46,14,12,2,69,21,18,3
1,DB00114,NUTR00045,Pyridoxal phosphate,54-47-7,0ABW0CG2BM,3-hydroxy-2-methyl-5-((phosphonooxy)methyl)-4-...,NGVDGCNFYWLIFO-UHFFFAOYSA-N,Cc1ncc(COP(=O)(O)O)c(C=O)c1O,8,1,6,0,12,2,9,1
2,DB00116,NUTR00056,Tetrahydrofolic acid,135-16-0,43ZWB253H4,"5,6,7,8-tetrahydrofolate | 5,6,7,8-tetrahydrof...",MSTNYGQPCMXVAQ-KIYNQFGBSA-N,Nc1nc(=O)c2c([nH]1)NCC(CNc1ccc(C(=O)N[C@@H](CC...,19,7,6,0,29,11,9,1
3,DB00117,NUTR00030,Histidine,71-00-1,4QD397987E,(S)-4-(2-Amino-2-carboxyethyl)imidazole | (S)-...,HNDVDQJCIGZPNO-YFKPBYRVSA-N,N[C@@H](Cc1c[nH]cn1)C(=O)O,6,3,2,0,9,5,3,1
4,DB00118,NUTR00052,Ademetionine,29908-03-0,7LP2MPO46S,Ademetionine | AdoMet | L-S-Adenosylmethionine...,MEFKEPWMEQBLKI-AIRLBKTGSA-N,C[S+](CC[C@H](N)C(=O)[O-])C[C@H]1O[C@@H](n2cnc...,15,6,5,1,23,9,8,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14327,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,37,1,5,0,56,2,8,1
14328,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0,48,1,15,1
14329,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,32,0,10,0,48,1,15,1
14330,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,29,7,1,1,44,11,2,2
